In [1]:
# Bước 1: Clone source
!git clone https://github.com/sangtran0897/OmniVoice.git
%cd OmniVoice
!pip install -e .

# Bước 2: Trỏ Python tới source folder
import sys
sys.path.insert(0, '/content/omnivoice')

f:\MyProjects\GIT\sangtran0897\Project\Tools\Forked\OmniVoice\notebooks\OmniVoice


Cloning into 'OmniVoice'...


Obtaining file:///F:/MyProjects/GIT/sangtran0897/Project/Tools/Forked/OmniVoice/notebooks/OmniVoice
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Checking if build backend supports build_editable: started
  Checking if build backend supports build_editable: finished with status 'done'
  Getting requirements to build editable: started
  Getting requirements to build editable: finished with status 'done'
  Installing backend dependencies: started
  Installing backend dependencies: finished with status 'done'
  Preparing editable metadata (pyproject.toml): started
  Preparing editable metadata (pyproject.toml): finished with status 'done'
  Building editable for omnivoice (pyproject.toml): started
  Building editable for omnivoice (pyproject.toml): finished with status 'done'
  Created wheel for omnivoice: filename=omnivoice-0.1.5-py3-none-any.whl size=11009 sha256=b15f0cb7698f90eb19d9e1e4652dc59bb48d5636f2db1798e7ca1ebe9d87d30d
  

In [2]:
from omnivoice import OmniVoice
import soundfile as sf
import torch
from IPython.display import Audio, display
import os
import shutil

# Model KhanhTTS trên Kaggle.
khanhtts_path = (
    "/kaggle/input/models/antontran1/"
    "khanhtts-omnivoice/pytorch/default/1"
)

# Audio tokenizer trên Kaggle.
audio_tokenizer_path = (
    "/kaggle/input/models/antontran1/"
    "higgs-audio-v2-tokenizer/pytorch/default/1"
)

# Thư mục model tổng hợp trong vùng có quyền ghi.
local_model_path = "/kaggle/working/KhanhTTS-OmniVoice-local"

# Xóa liên kết hoặc thư mục cũ nếu có.
if os.path.lexists(local_model_path):
    if os.path.islink(local_model_path):
        os.unlink(local_model_path)
    else:
        shutil.rmtree(local_model_path)

os.makedirs(local_model_path, exist_ok=True)

# Liên kết toàn bộ file và thư mục của KhanhTTS.
for item_name in os.listdir(khanhtts_path):
    source_path = os.path.join(khanhtts_path, item_name)
    destination_path = os.path.join(local_model_path, item_name)

    os.symlink(source_path, destination_path)

# Tạo đúng thư mục mà OmniVoice đang tìm.
os.symlink(
    audio_tokenizer_path,
    os.path.join(local_model_path, "audio_tokenizer")
)

print("Model chính:", khanhtts_path)
print("Audio tokenizer:", audio_tokenizer_path)
print("Model tổng hợp:", local_model_path)

# Nạp hoàn toàn từ đường dẫn Kaggle.
model = OmniVoice.from_pretrained(
    local_model_path,
    device_map="cuda:0",
    dtype=torch.float16,
    load_asr=False,
)

print("Đã nạp KhanhTTS và audio tokenizer từ Kaggle.")

f:\MyProjects\GIT\sangtran0897\Project\Tools\Forked\OmniVoice\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


FileNotFoundError: [WinError 3] The system cannot find the path specified: '/kaggle/input/models/antontran1/khanhtts-omnivoice/pytorch/default/1'

In [ ]:
# # from omnivoice import OmniVoice
# # from huggingface_hub import login
# # import soundfile as sf
# # import torch
# # from IPython.display import Audio, display
# # import os
# # import getpass

# # # Nhập token mới ở ô bảo mật.
# # # Khi nhập, token sẽ không hiện trên màn hình.
# # HF_TOKEN = ""

# # # Đăng nhập Hugging Face trước khi OmniVoice tải model.
# # login(token=HF_TOKEN, add_to_git_credential=False)

# # model = OmniVoice.from_pretrained(
# #     "sangtran0897/KhanhTTS-OmniVoice",
# #     device_map="cuda:0",
# #     dtype=torch.float16,
# #     load_asr=True,
# # )
# from omnivoice import OmniVoice
# import soundfile as sf
# import torch
# from IPython.display import Audio, display

# model_path = "/kaggle/input/models/antontran1/khanhtts-omnivoice/pytorch/default/1"

# model = OmniVoice.from_pretrained(
#     model_path,
#     device_map="cuda:0",
#     dtype=torch.float16,
#     load_asr=True,
# )

Loading weights:   0%|          | 0/313 [00:00<?, ?it/s]

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
from huggingface_hub import hf_hub_download

print("Downloading reference audio from Hugging Face...")

ref_audio_path = hf_hub_download(
    repo_id="sangtran0897/omnivoice-clone",
    filename="miennamchuan_5s.WAV",
    repo_type="dataset",   # quan trọng: repo này là dataset
    token=''
)

print(f"Loaded: {ref_audio_path}")


Loaded: /root/.cache/huggingface/hub/datasets--sangtran0897--omnivoice-clone/snapshots/3821845ea1267bba67c1e01b8df99ce7de5ade21/miennamchuan_5s.WAV


In [3]:
import ipywidgets as widgets
from IPython.display import display, Audio, clear_output
import soundfile as sf
import re
import subprocess
import numpy as np


# =========================
# Cấu hình
# =========================

SAMPLE_RATE = 24000

# Số giây nghỉ giữa mỗi audio sau khi gộp
# Ví dụ: audio 1 > nghi_cau > audio 2 > nghi_cau > audio 3
nghi_cau = 0.2

# Tốc độ audio sau khi generate xong
audio_speed = 1.05

ref_text = "để có thể đắp chút ánh hào quang rẻ tiền lên một gia tộc vốn đang khao khát có được sự chú ý."


# =========================
# Hàm xử lý audio
# =========================

def speedup_audio(input_path, output_path, speed=1.08):
    subprocess.run([
        "ffmpeg",
        "-y",
        "-i", input_path,
        "-filter:a", f"atempo={speed}",
        "-vn",
        output_path
    ], check=True)


def make_silence(seconds=0.18, sample_rate=24000):
    """
    Tạo khoảng lặng giữa các câu.
    """
    return np.zeros(int(seconds * sample_rate), dtype=np.float32)


def normalize_audio_array(audio_array):
    """
    Đảm bảo audio là numpy array 1 chiều float32.
    """
    audio_array = np.asarray(audio_array)

    if audio_array.ndim > 1:
        audio_array = audio_array[:, 0]

    return audio_array.astype(np.float32)


# =========================
# Hàm xử lý text
# =========================

def count_vietnamese_units(text):
    return len(re.findall(r"[A-Za-zÀ-ỹĐđ0-9]+", text))


def estimate_pause_seconds(text):
    comma = len(re.findall(r"[,，]", text)) * 0.12
    semi = len(re.findall(r"[;:；：]", text)) * 0.18
    end = len(re.findall(r"[.!?。！？]", text)) * 0.28
    newline = text.count("\n") * 0.35
    return comma + semi + end + newline


def estimate_duration_from_ref(text, ref_audio_path, ref_text, speed=1.0):
    ref_audio, sr = sf.read(ref_audio_path)
    ref_duration = len(ref_audio) / sr

    ref_units = max(count_vietnamese_units(ref_text), 1)
    target_units = max(count_vietnamese_units(text), 1)

    sec_per_unit = ref_duration / ref_units

    # Chặn biên để tránh audio mẫu có khoảng lặng làm duration bị lệch
    sec_per_unit = min(max(sec_per_unit, 0.22), 0.42)

    duration = target_units * sec_per_unit
    duration += estimate_pause_seconds(text)
    duration /= speed

    return round(max(duration, 1.2), 2)


def split_text_to_sentences(text):
    """
    Cắt text thành từng câu nhỏ.
    Cắt tại dấu chấm, chấm than, chấm hỏi.
    Giữ lại dấu câu ở cuối câu để TTS đọc tự nhiên hơn.
    """
    text = text.strip()

    # Gom nhiều khoảng trắng, nhiều dòng thành một khoảng trắng
    text = re.sub(r"\s+", " ", text)

    # Cắt theo . ! ? và cả dấu tương đương nếu có
    sentences = re.findall(r"[^.!?。！？]+[.!?。！？]+|[^.!?。！？]+$", text)

    # Làm sạch câu rỗng
    sentences = [s.strip() for s in sentences if s.strip()]

    return sentences


# =========================
# UI
# =========================

output = widgets.Output()

text_box = widgets.Textarea(
    value="",
    placeholder="Paste đoạn text vào đây...",
    description="Text:",
    layout=widgets.Layout(width="100%", height="300px")
)


# =========================
# Generate
# =========================

def on_text_change(change):
    if change["name"] == "value":
        input_text = change["new"].strip()

        if not input_text:
            return

        with output:
            clear_output(wait=True)
            print("Đang generate...")

            sentences = split_text_to_sentences(input_text)

            if not sentences:
                print("Không có câu nào để generate.")
                return

            print(f"Tổng số câu: {len(sentences)}")
            print(f"Nghỉ giữa mỗi câu: {nghi_cau}s")
            print(f"Speed audio sau cùng: {audio_speed}x")

            all_audio_segments = []

            for idx, sentence in enumerate(sentences, start=1):
                # print(f"\nĐang generate câu {idx}/{len(sentences)}:")
                # print(sentence)

                duration = estimate_duration_from_ref(
                    sentence,
                    ref_audio_path,
                    ref_text,
                    speed=1.2
                )

                # print(f"Estimated duration câu {idx}: {duration}s")

                audio = model.generate(
                    text=sentence,
                    ref_audio=ref_audio_path,
                    ref_text=ref_text,
                    duration=duration,
                    language="vi"
                )

                segment = normalize_audio_array(audio[0])
                all_audio_segments.append(segment)

                # Chèn nghỉ_câu giữa các audio, trừ audio cuối cùng
                if idx < len(sentences):
                    all_audio_segments.append(
                        make_silence(nghi_cau, SAMPLE_RATE)
                    )

            final_audio = np.concatenate(all_audio_segments)

            sf.write("clone_out.wav", final_audio, SAMPLE_RATE)

            # speedup_audio(
            #     "clone_out.wav",
            #     "clone_out_fast.wav",
            #     speed=audio_speed
            # )

            display(Audio("/kaggle/working/OmniVoice/OmniVoice/clone_out.wav"))

            print("\nXong.")
            print("File gốc: clone_out.wav")
            print("File đã tăng tốc: clone_out_fast.wav")


text_box.observe(on_text_change, names="value")

display(text_box, output)

ModuleNotFoundError: No module named 'ipywidgets'